В июле делал по этому скрипту, по идее проблем не должно быть. Классы заливаются сразу в БД и метчатся на уровне datalens

In [ ]:
import pandas as pd
import numpy as np
import csv
pd.set_option('display.max_colwidth', None)

### Загружаем файлы и сводим в один

In [ ]:
df = pd.read_excel(r"D:\Ricci\Продажи\Dataflat_July2026_M_NM_MO\realty_sold_05082026_M.xlsx", sheet_name="Данные")

In [ ]:
df.info()

In [ ]:
df2 = pd.read_excel(r"D:\Ricci\Продажи\Dataflat_July2026_M_NM_MO\realty_sold_05082026_MO.xlsx", sheet_name="Данные")

In [ ]:
df3 = pd.read_excel(r"D:\Ricci\Продажи\Dataflat_July2026_M_NM_MO\realty_sold_05082026_NM.xlsx", sheet_name="Данные")

In [ ]:
df_общий = pd.concat([df, df2, df3], ignore_index=True)

In [ ]:
df_общий.info()

In [ ]:
df_общий['ID дом.рф'] = df_общий['ID дом.рф'].apply(lambda x: str(int(x)) if pd.notna(x) else '')
df_общий['ID ЖК'] = df_общий['ID ЖК'].apply(lambda x: str(int(x)) if pd.notna(x) else '')


In [ ]:
df.isna().sum()

In [ ]:
# Сначала убедимся, что все столбцы в числовом формате
# (пустые строки станут NaN)
for col in ['Оценка цены', 'Цена со скидкой', 'Оценка по ЕИСЖС']:
    df_общий[col] = pd.to_numeric(df_общий[col], errors='coerce')

# Заполняем пропуски: сначала из Цена со скидкой, затем из Оценка по ЕИСЖС
df_общий['Оценка цены'] = df_общий['Оценка цены'].fillna(df_общий['Цена со скидкой'])
df_общий['Оценка цены'] = df_общий['Оценка цены'].fillna(df_общий['Оценка по ЕИСЖС'])

In [ ]:
df_общий['Уступка'].unique()

In [ ]:
df_общий["Ипотека"].unique()

In [ ]:
df_общий = df_общий[df_общий['Тип помещения'].isin(['квартира', 'апартамент'])]

In [ ]:
yearly_counts = df_общий['Дата регистрации'].dt.year.value_counts().sort_index()
print(yearly_counts)

In [ ]:
df_общий = df_общий[df_общий['Дата регистрации'] > '2016-01-01']

Убираем строки с покупателем ЮЛ, где покупатель купил более 5 лотов, лоты от 'Московский фонд реновации', а также лоты с уступкой

In [ ]:
df_общий = df_общий[(df_общий['Покупатель ЮЛ'].isna()) | (df_общий['Покупатель ЮЛ'] == '')]

In [ ]:
df_общий = df_общий[df_общий['Купил лотов в ЖК'] <= 5]

In [ ]:
df_общий = df_общий[df_общий['Уступка'] == 0]

In [ ]:
df_общий = df_общий[df_общий['Застройщик ЖК'] != 'Московский фонд реновации']

In [ ]:
# Извлекаем год в новый столбец
df_общий['Дата регистрации'] = pd.to_datetime(df_общий['Дата регистрации'])
df_общий['Год'] = df_общий['Дата регистрации'].dt.year
df_общий['Месяц'] = df_общий['Дата регистрации'].dt.month

Проверим число пропусков в столбце 'Оценка цены' по месяцам

In [ ]:
empty_count = df_общий['Оценка цены'].isna().sum() + (df_общий['Оценка цены'] == '').sum() + (df_общий['Оценка цены'].isnull()).sum()
print('Количество пропусков в столбце Оценка цены:', empty_count)

In [ ]:
pivot_empty = df_общий[df_общий['Оценка цены'].isna()].pivot_table(
    index='Год',
    columns='Месяц',
    aggfunc='size',
    fill_value=0
)
print(pivot_empty)


Заполнение пропусков цен с помощью средней цены за метр:

In [ ]:
# Сначала преобразуем столбец в числовой тип (пустые строки станут NaN)
df_общий['Площадь'] = pd.to_numeric(df_общий['Площадь'], errors='coerce')
has_missing = df_общий['Площадь'].isna().any()
print(f"Есть пропуски в столбце 'Площадь': {has_missing}")
missing_count = df_общий['Площадь'].isna().sum()
print(f"Количество пропусков в столбце 'Площадь': {missing_count}")

In [ ]:
# Удаляем строки с NaN
df_общий = df_общий.dropna(subset=['Площадь'])

In [ ]:
# Рассчитываем только там, где оба значения не пустые
df_общий['Цена за метр'] = df_общий['Оценка цены'] / df_общий['Площадь']

In [ ]:
# 2. Заполняем средним по полной группе (ЖК + Год + Месяц + Комнатность)
df_общий['Цена за метр'] = df_общий.groupby(['ЖК рус', 'Год', 'Месяц', 'Тип Комнатности'])['Цена за метр'].transform(
    lambda x: x.fillna(x.mean())
)

# 3. Если остались пропуски, заполняем средним по группе без комнатности
df_общий['Цена за метр'] = df_общий.groupby(['ЖК рус', 'Год', 'Месяц'])['Цена за метр'].transform(
    lambda x: x.fillna(x.mean())
)

# 4. Если всё еще есть пропуски, заполняем средним по ЖК
df_общий['Цена за метр'] = df_общий.groupby(['ЖК рус'])['Цена за метр'].transform(
    lambda x: x.fillna(x.mean())
)

print("Пропуски успешно заполнены!")
print(f"Осталось пропусков: {df_общий['Цена за метр'].isna().sum()}")

In [ ]:
df_общий['Оценка цены'] = df_общий['Оценка цены'].fillna(df_общий['Цена за метр'] * df_общий['Площадь']).round(0)

In [ ]:
df_общий = df_общий.dropna(subset=['Оценка цены']).reset_index(drop=True)

In [ ]:
df_общий['Цена за метр'] = df_общий['Цена за метр'].round(1)

In [ ]:
df_общий.info()

In [ ]:
import json

In [ ]:
with open(r'C:\PycharmProjects\ndv_parcing\!haracteristik_dictionary\projects.json', 'r', encoding='utf-8') as f: json_data = json.load(f)

In [ ]:
# 1. Создаем словарь для маппинга: ID ЖК -> (Название проекта, Девелопер)
mapping = {}
for project_name, project_info in json_data.items():
    project_id = project_info.get('id')
    developer = project_info.get('Девелопер')

    # Пропускаем, если id = 'nan' или None
    if project_id and project_id != 'nan' and pd.notna(project_id):
        mapping[project_id] = {
            'Название проекта': project_name,
            'Девелопер': developer
        }

In [ ]:
# 2. Создаем столбцы и заполняем
df_общий['Название проекта'] = df_общий['ID ЖК'].map(lambda x: mapping.get(x, {}).get('Название проекта', ''))
df_общий['Девелопер'] = df_общий['ID ЖК'].map(lambda x: mapping.get(x, {}).get('Девелопер', ''))

In [ ]:
# df_общий['Тип Комнатности пыпин'] = df_общий['Тип Комнатности']

In [ ]:
df_общий = df_общий.rename(columns={'Тип Комнатности': 'Кол-во комнат'})

In [ ]:
df_общий['ЖК рус'].unique()
# нужный порядок столбцов
columns_order = [
    "ID ЖК",
    "ЖК рус",
    "Район Город",
    "Округ Направление",
    "Регион",
    "АТД",
    "Застройщик ЖК",
    "Площадь",
    "Тип Комнатности",
    "Тип помещения",
    "Корпус",
    "Дата регистрации",
    "Залогодержатель",
    "Тип обременения",
    "Оценка цены",
    "Ипотека",
    'Цена за метр',
    "ID дом.рф",
    'Название проекта',
    'Девелопер'
]

# оставляем и упорядочиваем столбцы
df_общий = df_общий.reindex(columns=columns_order)

Заполнение по файлу с квартирографией

In [ ]:
json_path = r'C:\PycharmProjects\ndv_parcing\area_dictionary\output.json' # база квартирографии

def load_json_data(json_path):
    with open(json_path, 'r', encoding='utf-8') as file:
        return json.load(file)


def load_excel_data(excel_path):
    df = pd.read_csv(excel_path)
    df.columns = df.columns.str.strip()
    return df


def process_data(json_data, df_общий):
    result_df = df_общий.copy()
    result_df['Площадь'] = result_df['Площадь'].astype(str).str.replace(',', '.').str.replace(' ', '').astype(float)

    total = len(result_df)

    # Список застройщиков, у которых не надо менять типологию
    developers_to_skip = {'фонд реновации'}
    # Список проектов, у которых не надо менять типологию
    jk_name_to_skip = {'гармония парк', 'мишино-2'}
    jk_name_to_skip2 = {'серебро', 'берег'}

    for idx, row in result_df.iterrows():

        jk_name = str(row.get('Название проекта')).strip()
        area = row.get('Площадь')
        developer = str(row.get('Девелопер')).strip()

        if pd.isna(jk_name) or pd.isna(area):
            result_df.at[idx, 'Кол-во комнат'] = 'Н/Д'
            continue

        # Условие: если площадь <= 28 — это студия
        if area <= 28 and jk_name not in jk_name_to_skip2:
            result_df.at[idx, 'Кол-во комнат'] = 'студия'
            if __name__ == "__main__":
                print(f"[{idx + 1}/{total}] Назначено как студия по площади <= 28: ЖК {jk_name}, площадь {area}")
            continue


        found = False

        if jk_name in json_data:
            jk_dict = json_data[jk_name]
            area = round(float(area), 2)

            # Ищем точное совпадение
            for json_area_str, room_type in jk_dict.items():
                try:
                    json_area = round(float(json_area_str), 2)
                    if area == json_area:
                        result_df.at[idx, 'Кол-во комнат'] = (
                            'студия' if room_type == 0 or
                            (isinstance(room_type, str) and (
                                'ст' in room_type.lower() or
                                room_type.strip().lower() == 'st' or
                                'СТ' in room_type
                            ))
                            else room_type
                        )
                        found = True
                        break
                except ValueError:
                    continue

            if not found:
                closest_area = None
                closest_room = None

                # Ищем ближайшее СНИЗУ
                for json_area_str, room_type in jk_dict.items():
                    try:
                        json_area = round(float(json_area_str), 2)
                        if area - 3 <= json_area < area:
                            if closest_area is None or json_area > closest_area:
                                closest_area = json_area
                                closest_room = room_type
                    except ValueError:
                        continue

                # Если не нашли — ищем СВЕРХУ
                if closest_area is None:
                    for json_area_str, room_type in jk_dict.items():
                        try:
                            json_area = round(float(json_area_str), 2)
                            if area < json_area <= area + 3:
                                if closest_area is None or json_area < closest_area:
                                    closest_area = json_area
                                    closest_room = room_type
                        except ValueError:
                            continue

                if closest_area is not None:
                    result_df.at[idx, 'Кол-во комнат'] = (
                        'студия' if closest_room == 0 or
                        (isinstance(closest_room, str) and (
                            'ст' in closest_room.lower() or
                            closest_room.strip().lower() == 'st' or
                            'СТ' in closest_room
                        ))
                        else closest_room
                    )
                    found = True


        if __name__ == "__main__":
            print(f"[{idx + 1}/{total}] Обработано: ЖК {jk_name}, площадь {area}")

    return result_df

In [ ]:
json_data = load_json_data(json_path)
result_df = process_data(json_data, df_общий)

In [ ]:
result_df.info()

In [ ]:
result_df['Ипотека'].unique()

In [ ]:
result_df['ID дом.рф'].unique()

In [ ]:
result_df["Ипотека"] = result_df["Ипотека"].astype("Int64").astype(str)

In [ ]:
result_df['Кол-во комнат'].unique()

In [ ]:
# нужный порядок столбцов
columns_order = [
    "ID ЖК",
    "ЖК рус",
    "Район Город",
    "Округ Направление",
    "Регион",
    "АТД",
    "Застройщик ЖК",
    "Площадь",
    "Кол-во комнат",
    "Тип помещения",
    "Корпус",
    "Дата регистрации",
    "Залогодержатель",
    "Тип обременения",
    "Оценка цены",
    "Ипотека",
    'Цена за метр',
    "ID дом.рф",
    'Название проекта',
    'Девелопер'
]

# оставляем и упорядочиваем столбцы
result_df = result_df.reindex(columns=columns_order)

In [ ]:
result_df.to_csv(r"D:\Ricci\Продажи\Pipin-08-2026.csv", index=False, encoding='utf-8-sig')